In [7]:
from pathlib import Path

In [8]:
PROJECT_ROOT = Path.cwd().parent

DATA_ROOT = PROJECT_ROOT / "data/sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
        DATA_ROOT
        / "processed"
        / "stage_7_processed_dataset"
        / SAMPLE_ID
)

FEATURE_DIR = PROCESSED_DIR / "cells"

In [9]:
import pandas as pd
import json
from pathlib import Path

output_dir = Path("../data/sample/processed/stage_9_track_stitching")  # Change this to your output directory

# ------------------------------------------------------------
# Load detections
# ------------------------------------------------------------
detections_df = pd.read_csv(output_dir / "detections.csv")

# If you want the original list of DataFrames (time_frames)
time_frames = [
    frame_df.reset_index(drop=True)
    for _, frame_df in detections_df.groupby("frame")
]

# ------------------------------------------------------------
# Load tracks
# ------------------------------------------------------------
tracks = pd.read_csv(output_dir / "tracks.csv")

# ------------------------------------------------------------
# Load segmentation events
# ------------------------------------------------------------
segmentation_events = pd.read_csv(output_dir / "segmentation_events.csv")

# ------------------------------------------------------------
# Load track endings
# ------------------------------------------------------------
track_endings = pd.read_csv(output_dir / "track_endings.csv")

# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------
with open(output_dir / "metadata.json", "r") as f:
    metadata = json.load(f)

print("Loaded all Stage 7 results.")

Loaded all Stage 7 results.


In [10]:
import zarr

ZARR_PATH = (
        DATA_ROOT
        / "biohub_5samples_20timepoints"
        / "train"
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
)

ARRAY_PATH = ZARR_PATH / "0"

original_volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)

In [11]:
# ------------------------------------------------------------
# Convert tracks to Napari format
# ------------------------------------------------------------

tracks_array = tracks[
    ["track_id", "frame", "z", "y", "x"]
].to_numpy(dtype=float)

points_array = tracks[
    ["frame", "z", "y", "x"]
].to_numpy(dtype=float)

track_ids = tracks["track_id"].to_numpy()

**Broken Tracks**

Red cells -> Tracks that will break before the last frame

Green cells -> Tracks that will born after the first frame

In [13]:
import napari
import numpy as np

VOXEL_SIZE = (
    1.625,      # Z
    0.40625,    # Y
    0.40625,    # X
)

# ------------------------------------------------------------
# Find new and ended tracks
# ------------------------------------------------------------

first_frame = tracks["frame"].min()
last_frame = tracks["frame"].max()

track_summary = (
    tracks.groupby("track_id")["frame"]
    .agg(first_frame="min", last_frame="max")
    .reset_index()
)

new_track_ids = track_summary.loc[
    track_summary["first_frame"] > first_frame,
    "track_id",
]

ended_track_ids = track_summary.loc[
    track_summary["last_frame"] < last_frame,
    "track_id",
]

new_tracks = tracks[
    tracks["track_id"].isin(new_track_ids)
]

ended_tracks = tracks[
    tracks["track_id"].isin(ended_track_ids)
]

# ------------------------------------------------------------
# Launch Napari
# ------------------------------------------------------------

viewer = napari.Viewer(ndisplay=3)

# ------------------------------------------------------------
# Broken tracks
# ------------------------------------------------------------

viewer.add_image(
    original_volume,
    name="Raw Volume",
    scale=(1, *VOXEL_SIZE),
    rendering="mip",
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
)

# Ended tracks (red)
viewer.add_tracks(
    ended_tracks[
        ["track_id", "frame", "z", "y", "x"]
    ].to_numpy(float),
    name="Ended Tracks",
    scale=(1, *VOXEL_SIZE),
    tail_length=20,
)

viewer.add_points(
    ended_tracks[
        ["frame", "z", "y", "x"]
    ].to_numpy(float),
    name="Ended Centroids",
    scale=(1, *VOXEL_SIZE),
    size=4,
    face_color="red",
)

# New tracks (green)
viewer.add_tracks(
    new_tracks[
        ["track_id", "frame", "z", "y", "x"]
    ].to_numpy(float),
    name="New Tracks",
    scale=(1, *VOXEL_SIZE),
    tail_length=20,
)

viewer.add_points(
    new_tracks[
        ["frame", "z", "y", "x"]
    ].to_numpy(float),
    name="New Centroids",
    size=4,
    scale=(1, *VOXEL_SIZE),
    face_color="lime",
)

# ------------------------------------------------------------
# All-original
# ------------------------------------------------------------

# Raw microscopy image
viewer.add_image(
    original_volume,
    name="Raw Volume - all",
    rendering="mip",          # Try "attenuated_mip" as well
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
    scale=(1, *VOXEL_SIZE),
).visible = False

# Track trajectories
viewer.add_tracks(
    tracks_array,
    name="Tracks - all",
    tail_length=20,
    scale=(1, *VOXEL_SIZE),
).visible = False

# Cell centroids
viewer.add_points(
    points_array,
    name="Centroids - all",
    size=4,
    face_color="red",
    properties={
        "track_id": track_ids,
    },
    scale=(1, *VOXEL_SIZE),
).visible = False

print(f"New tracks:   {len(new_track_ids)}")
print(f"Ended tracks: {len(ended_track_ids)}")

napari.run()

New tracks:   196
Ended tracks: 174
